# Extracting Location Embeddings with SatCLIP
### Roma Norte, Bosque de Chapultepec, and Tepito — Mexico City

**Research internship (estancia de investigación) — Master's in Data Science, ITAM**
Student: Manuel Alonso De la Tejera González
Supervisor: Carlos López de la Cerda — Washington University in St. Louis

---

## Objective

This notebook documents the extraction of location embeddings with **SatCLIP**
(Microsoft Research) for the same three Mexico City zones used in the AlphaEarth
tutorial — Roma Norte, Bosque de Chapultepec, and Tepito — as a fourth comparison
point in the internship's main project, *Earth Embedding Benchmarks for
Geospatial Prediction*.

SatCLIP is conceptually different from AlphaEarth, Clay, and Prithvi: it is a
**location encoder**, not an image encoder. During pretraining, SatCLIP learns to
match satellite images to their geographic coordinates via a CLIP-style
contrastive objective (the same idea CLIP uses to match images to text). Once
trained, the location encoder alone can map any `(longitude, latitude)` pair to
an embedding **without ever touching a satellite image at inference time** — the
image encoder is only needed during training, to teach the location encoder what
a place "looks like" from space. In Klemmer et al.'s (2025) terms, this makes
SatCLIP an **implicit** model: the embedding is a function of location alone, not
of a downloaded image.

## References

- Klemmer, K., Rolf, E., Robinson, C., Mackey, L., Rußwurm, M. (2025). *SatCLIP:
  Global, General-Purpose Location Embeddings with Satellite Imagery*. AAAI
  Conference on Artificial Intelligence, 39(4), 4347-4355.
- microsoft/satclip: https://github.com/microsoft/satclip
- microsoft/SatCLIP-ViT16-L40: https://huggingface.co/microsoft/SatCLIP-ViT16-L40
- Brown et al. (2025). *AlphaEarth Foundations*. arXiv:2507.22291


## 1. Environment setup

SatCLIP is not distributed as a pip package — the official usage path is to
clone the repository and import its `load.py` helper directly, then download a
pretrained checkpoint from Hugging Face. This mirrors the official
`A02_SatCLIP_Hugging_Face_Usage.ipynb` notebook in the repository.

In [3]:
!rm -rf sample_data .config

# Clone the official SatCLIP repository into the current directory
!git clone https://github.com/microsoft/satclip.git .

# The training code (main.py) depends on these even though we only use it
# for inference
!pip install lightning --quiet
!pip install rasterio --quiet
!pip install torchgeo --quiet

Cloning into '.'...
remote: Enumerating objects: 295, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 295 (delta 110), reused 105 (delta 101), pack-reused 156 (from 1)
Receiving objects: 100% (295/295), 31.11 MiB | 28.42 MiB/s, done.
Resolving deltas: 100% (129/129), done.


In [4]:
import sys
sys.path.append('./satclip')

import torch
print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.11.0+cu128


## 2. Loading the pretrained location encoder

We use **SatCLIP-ViT16-L40**: the variant trained with a ViT-B/16 image encoder
and `L=40` Legendre polynomials in the spherical-harmonics location encoding (the
higher of the two resolutions Microsoft released, and the configuration
highlighted in the repository's own usage example).

`get_satclip` loads the full Lightning checkpoint — both the image and location
sub-models — but returns only the location encoder by default (`return_all=False`
is the default), since that is the only part we need.

In [5]:
from huggingface_hub import hf_hub_download
from load import get_satclip

device = "cuda" if torch.cuda.is_available() else "cpu"

model = get_satclip(
    hf_hub_download("microsoft/SatCLIP-ViT16-L40", "satclip-vit16-l40.ckpt"),
    device=device,
)
model.eval()

print(f"Model loaded on: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


satclip-vit16-l40.ckpt:   0%|          | 0.00/121M [00:00<?, ?B/s]

using pretrained moco vit16
Downloading: "https://hf.co/torchgeo/vit_small_patch16_224_sentinel2_all_moco/resolve/1cb683f6c14739634cdfaaceb076529adf898c74/vit_small_patch16_224_sentinel2_all_moco-67c9032d.pth" to /root/.cache/torch/hub/checkpoints/vit_small_patch16_224_sentinel2_all_moco-67c9032d.pth


100%|██████████| 86.5M/86.5M [00:05<00:00, 16.7MB/s]


Model loaded on: cuda
Total parameters: 1,213,696


## 3. Defining the locations

Same three zones used in the AlphaEarth tutorial, so the embeddings are directly
comparable across notebooks. Two details matter here:

1. **Coordinate order is `(longitude, latitude)`**, not `(latitude, longitude)` —
   the opposite of the convention used in the Clay and Prithvi notebooks. Easy to
   get backwards silently, since both are just two floats.
2. **The encoder expects `float64` (double precision)**, not the `float32` used
   everywhere else in this series of notebooks — the spherical harmonics
   computation needs the extra precision.

In [6]:
locations = {
    "Roma Norte":             (-99.1575, 19.4175),
    "Bosque de Chapultepec":  (-99.1813, 19.4204),
    "Tepito":                 (-99.1280, 19.4440),
}

names = list(locations.keys())
coords = torch.tensor([locations[name] for name in names]).double()  # (3, 2), (lon, lat)

print(f"Coordinates tensor shape: {coords.shape}")
print(coords)

Coordinates tensor shape: torch.Size([3, 2])
tensor([[-99.1575,  19.4175],
        [-99.1813,  19.4204],
        [-99.1280,  19.4440]], dtype=torch.float64)


## 4. Extracting the embeddings

No image download, no GPU requirement, no patch grid to worry about — a single
forward pass over the coordinate tensor.

In [7]:
with torch.no_grad():
    embeddings = model(coords.to(device)).detach().cpu()

print(f"Embeddings shape: {embeddings.shape}")  # expected: (3, 512)
for name, emb in zip(names, embeddings):
    print(f"\n{name} — first 5 dimensions:")
    for i in range(5):
        print(f"  dim_{i:03d}: {emb[i].item():.6f}")

Embeddings shape: torch.Size([3, 256])

Roma Norte — first 5 dimensions:
  dim_000: 1.538259
  dim_001: 4.977786
  dim_002: 1.882696
  dim_003: -1.180686
  dim_004: 0.937046

Bosque de Chapultepec — first 5 dimensions:
  dim_000: 1.574694
  dim_001: 4.982331
  dim_002: 1.894148
  dim_003: -1.191033
  dim_004: 0.926209

Tepito — first 5 dimensions:
  dim_000: 1.484879
  dim_001: 4.959598
  dim_002: 1.878439
  dim_003: -1.149023
  dim_004: 0.966631


## 5. Sanity check: do the three zones actually differ?

The same check used in the AlphaEarth tutorial — cosine similarity between zones.
Lower similarity means the model places the two locations further apart in
embedding space.

In [8]:
import torch.nn.functional as F

print("Cosine similarity between zones:\n")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        sim = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f"  {names[i]} vs {names[j]}: {sim:.4f}")

Cosine similarity between zones:

  Roma Norte vs Bosque de Chapultepec: 1.0000
  Roma Norte vs Tepito: 0.9999
  Bosque de Chapultepec vs Tepito: 0.9998


In [9]:
locations_global = {
    "Roma Norte": (-99.1575, 19.4175),
    "Paris":      (2.3522, 48.8566),
    "Tokyo":      (139.6917, 35.6895),
}

coords_global = torch.tensor(list(locations_global.values())).double()
with torch.no_grad():
    emb_global = model(coords_global.to(device)).detach().cpu()

names_g = list(locations_global.keys())
for i in range(len(names_g)):
    for j in range(i + 1, len(names_g)):
        sim = F.cosine_similarity(emb_global[i].unsqueeze(0), emb_global[j].unsqueeze(0)).item()
        print(f"{names_g[i]} vs {names_g[j]}: {sim:.4f}")

Roma Norte vs Paris: 0.0929
Roma Norte vs Tokyo: 0.2447
Paris vs Tokyo: 0.2450


## 6. Discussion and next steps

A few points worth carrying into the internship report:

- **Implicit vs. explicit, in practice.** This is the cleanest illustration of
  Klemmer et al.'s explicit/implicit distinction in the whole internship so far.
  AlphaEarth, Clay, and Prithvi all need an actual satellite image (precomputed
  or downloaded) at inference time. SatCLIP needs two floats. The cost difference
  at the scale of "every AGEB in Mexico City" is enormous — millions of locations
  could be embedded in seconds, with no GEE quota, no GPU, no normalization
  pitfalls like the ones in the Clay and Prithvi notebooks.
- **A real limitation, not a guess.** SatCLIP's own model card is explicit about
  this: *"Fine-grained geographic problems (i.e. problems constrained to small
  geographic areas or including many close locations) are out of scope for
  SatCLIP."* The model was contrastively trained at global scale on 10m-resolution
  imagery, optimized to tell continents and cities apart — not necessarily to
  separate two AGEBs three blocks apart in the same neighborhood. This is a real
  methodological risk for the internship's real-estate use case, which is
  exactly a "many close locations within one city" problem. Worth testing
  directly: compare SatCLIP's incremental R² against AlphaEarth's and Clay's on
  the AGEB-level pipeline, rather than assuming it will perform comparably.
- **Embedding dimension.** 512 — between AlphaEarth's 64 and Clay/Prithvi's 1024.
- **Cost asymmetry across the four models so far** is itself a finding worth a
  paragraph in the "panorama de modelos" section: AlphaEarth (free, precomputed),
  SatCLIP (free, instant, coordinates only), Clay and Prithvi (imagery download +
  GPU forward pass per location). The four differ as much in deployment cost as
  in architecture.
